In [ ]:
import sys
print(sys.executable)

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import matplotlib.ticker as mtick
from pathlib import Path


# Load data
df = pd.read_csv("../data/processed/rbm_feature_engineered_trials.csv")

# Results path for figures
RESULTS = Path("../results/figures")
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
# Check what's in the dataframe
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print(df.head())

In [ ]:
# --- Figure 1: Distribution of RBM scores ---

score_counts = df["total_risk_score"].value_counts().sort_index()

# force full range (1-10)
all_scores = pd.Series(0, index=range(1, 11), dtype=float)
score_counts = all_scores.add(score_counts, fill_value=0)

plt.figure()

score_counts.plot(
    kind="bar",
    width=0.85,
    edgecolor="black"
)

plt.title("Fig 1: Distribution of RBM Risk Scores (n ≈ 5000)")
plt.xlabel("RBM Score")
plt.ylabel("Number of Trials")

plt.xticks(rotation=0)

plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(100))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(50))

plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()

plt.savefig(
    RESULTS / "fig1_rbm_score_distribution.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.2
    )
plt.show()

In [ ]:
print("order exists:", "order" in globals())

In [ ]:
# --- Figure 2: Distribution of RBM scores by intervention type ---

# Clean intervention type labels
df["intervention_type"] = (
    df["intervention_type"]
    .str.replace("_", " ")
    .str.title()
    .str.strip()
)

# Order by mean risk score (descending)
order = (
    df.groupby("intervention_type")["total_risk_score"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

plt.figure()

ax = sns.boxplot(
    data=df,
    x="intervention_type",
    y="total_risk_score",
    order=order,
    color="#4C72B0",
    width=0.6,
    linewidth=1.5,
    medianprops=dict(color="black", linewidth=2.5),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    fliersize=4
)

plt.title("Figure 2: RBM Risk Score Distribution by Intervention Type (n ≈ 5000 Trials)", fontsize=14, pad=15)
plt.xlabel("Intervention Type", fontsize=12)
plt.ylabel("RBM Risk Score", fontsize=12)

ax.set_ylim(0.8, 10.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

plt.xticks(rotation=45, ha="right", fontsize=11)
ax.tick_params(axis="y", labelsize=11)

ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.grid(axis="x", visible=False)

plt.tight_layout()

plt.savefig(
    RESULTS / "fig2_intervention_vs_risk.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.2
    )
plt.show()

In [ ]:
# --- Figure 3: Phase vs RBM score ---

# Clean phase labels 
df["phase_clean"] = (
    df["phase"]
    .replace({
        "EARLY_PHASE1": "Early Phase 1",
        "PHASE1": "Phase 1",
        "PHASE2": "Phase 2",
        "PHASE3": "Phase 3",
        "PHASE4": "Phase 4",
        "PHASE_1": "Phase 1",
        "PHASE_2": "Phase 2",
        "PHASE_3": "Phase 3",
        "PHASE_4": "Phase 4",
        "NA": "Unknown",
        "N/A": "Unknown",
        "UNKNOWN": "Unknown",
        None: "Unknown"
    })
    .astype(str)
    .str.strip()
)

# Establish clinical order 
phase_order = [
    "Early Phase 1",
    "Phase 1",
    "Phase 2",
    "Phase 3",
    "Phase 4",
    "Unknown"
]

plt.figure()

sns.boxplot(
    data=df,
    x="phase_clean",
    y="total_risk_score",
    order=phase_order,
    linewidth=1,
    fliersize=3
)

plt.title("Fig 3: RBM Risk Score Distribution by Clinical Trial Phase (n ≈ 5000 Trials)")
plt.xlabel("Trial Phase")
plt.ylabel("RBM Risk Score")

# Fixed y axis
plt.ylim(1, 10.5)
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(1))

plt.xticks(ha="center")

plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()

plt.savefig(
    RESULTS / "fig3_phase_vs_risk.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.2
    )
plt.show()

In [ ]:
# --- Figure 4: Enrollment size vs RBM score ---

sns.set_style("whitegrid")

# Clean data (remove zero enrollment trials)
df_plot = df[df["enrollment"] > 0].copy()

# Bin enrollment into quintiles for better visualization 
df_plot["enrollment_bin"] = pd.qcut(
    df_plot["enrollment"],
    q=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"]
)

plt.figure(figsize=(8, 5))

ax = sns.boxplot(
    data=df_plot,
    x="enrollment_bin",
    y="total_risk_score",
    color="#4C72B0",     
    width=0.6,
    showfliers=False     
)

plt.title("Fig 4: RBM Risk Score Distribution by Enrollment Size (n ≈ 5000 Trials)", fontsize=14)
plt.xlabel("Enrollment Size (Quintiles)", fontsize=12)
plt.ylabel("RBM Risk Score", fontsize=12)

# Fix y axis
plt.ylim(1, 10.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

plt.xticks(rotation=0)

ax.tick_params(axis="x", labelsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.grid(axis="x", visible=False)

plt.tight_layout()

plt.savefig(
    RESULTS / "fig4_enrollment_risk.png",
     dpi=300,
    bbox_inches="tight",
    pad_inches=0.2
    )
plt.show()

In [ ]:
# --- Exploratory Figure A: RBM Risk Composition by Enrollment Size ---

# Check if risk_category already exists to avoid overwriting
if "risk_category" not in df.columns:
    df["risk_category"] = pd.cut(
        df["total_risk_score"],
        bins=[0, 4, 7, 10],
        labels=["Low Risk", "Medium Risk", "High Risk"]
    )

# Clean data
df_plot = df[df["enrollment"] > 0].copy()

# Bin enrollment into quintiles for better visualization
df_plot["enrollment_bin"] = pd.qcut(
    df_plot["enrollment"],
    q=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"]
)

# Create risk categories
df_plot["risk_category"] = pd.cut(
    df_plot["total_risk_score"],
    bins=[0, 4, 7, 10],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

df_plot = df_plot.dropna(subset=["enrollment_bin", "risk_category"])

# Create proportions table
count_df = (
    df_plot
    .groupby(["enrollment_bin", "risk_category"], observed=True)
    .size()
    .unstack(fill_value=0)
)

prop_df = count_df.div(count_df.sum(axis=1), axis=0)

risk_order = ["High Risk", "Medium Risk", "Low Risk"]
prop_df = prop_df[risk_order]

colors = {
    "Low Risk": "#2ECC71",
    "Medium Risk": "#F39C12",
    "High Risk": "#E74C3C"
}

ax = prop_df.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=[colors[r] for r in risk_order],
    width=0.7
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_locator(mtick.MultipleLocator(0.1))  

plt.ylim(0, 1)

plt.title(
    "Exploratory A: RBM Risk Composition by Enrollment Size (n ≈ 5000 Trials)",
    fontsize=14
)
plt.xlabel("Enrollment Size (Quintiles)", fontsize=12)
plt.ylabel("Percentage of Trials", fontsize=12)

ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")
ax.tick_params(axis='both', labelsize=11)

# Legend
plt.legend(
    title="RBM Risk Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=10,
    title_fontsize=11
)

ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.grid(axis="x", visible=False)

plt.tight_layout()

# Create exploratory subdirectory parallel to figures (within results)
(RESULTS.parent / "exploratory").mkdir(parents=True, exist_ok=True)

plt.savefig(
    RESULTS.parent / "exploratory/exploratory_A_enrollment_risk.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.2
    )
plt.show()